In [1]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report


In [2]:
df = pd.read_csv('hotel_bookings.csv')

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.describe()

In [ ]:
df.describe(include='object')

In [ ]:
df.isnull().sum()

In [ ]:
df["children"].fillna(0,inplace=True)

In [ ]:

df["agent"].fillna(0,inplace=True)


In [ ]:
df["company"].fillna(0,inplace=True)


In [ ]:
df["country"].fillna("Unknown",inplace=True)

In [ ]:
df.duplicated().sum()

In [3]:
leakage_cols = ['reservation_status', 'reservation_status_date', 'assigned_room_type', 'company']
df.drop(columns=leakage_cols, inplace=True)


In [4]:
df['children'] = df['children'].fillna(0)
total_guests = df['adults'] + df['children'] + df['babies']
df = df[(total_guests > 0) & (df['adr'] >= 0)].copy()

In [5]:
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)


In [6]:
df['total_nights'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']
df['total_guests'] = df['adults'] + df['children'] + df['babies']
df['is_family'] = ((df['children'] + df['babies']) > 0).astype(int)
df['had_prev_cancel'] = (df['previous_cancellations'] > 0).astype(int)
df['adr_per_guest'] = df['adr'] / df['total_guests']

In [7]:
df['agent'] = df['agent'].fillna(0).astype(int).astype(str)
X = df.drop(columns=['is_canceled'])
y = df['is_canceled'].values


In [8]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)


In [9]:
cap_cols = ['lead_time', 'adr', 'adr_per_guest', 'total_nights']
for col in cap_cols:
    upper_limit = X_train[col].quantile(0.99)
    X_train[col] = X_train[col].clip(upper=upper_limit)
    X_val[col] = X_val[col].clip(upper=upper_limit)
    X_test[col] = X_test[col].clip(upper=upper_limit)


In [10]:
for dataset in [X_train, X_val, X_test]:
    dataset['lead_time'] = np.log1p(dataset['lead_time'])


In [11]:
top_countries = X_train['country'].value_counts()
common_countries = set(top_countries[top_countries >= 200].index)
for dataset in [X_train, X_val, X_test]:
    dataset['country'] = dataset['country'].fillna('Unknown')
    dataset['country'] = dataset['country'].where(dataset['country'].isin(common_countries), 'Other')


In [12]:
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()

num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_cols),
    ('cat', cat_transformer, cat_cols)
])


C:\Users\ahmed\AppData\Local\Temp\ipykernel_2340\407624714.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()


In [13]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Tuned Decision Tree": DecisionTreeClassifier(max_depth=12, min_samples_leaf=20, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=150, max_depth=15, random_state=42, n_jobs=-1),
    "LightGBM": LGBMClassifier(n_estimators=300, learning_rate=0.05, random_state=42, verbose=-1)
}

results = []
trained_pipelines = {}

for name, clf in models.items():
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', clf)
    ])
    pipe.fit(X_train, y_train)
    trained_pipelines[name] = pipe
    
    # التقييم على بيانات التحقق (Validation Set) لاختيار الأفضل
    val_preds = pipe.predict(X_val)
    results.append({
        "Model": name,
        "Val Accuracy": accuracy_score(y_val, val_preds),
        "Val Precision": precision_score(y_val, val_preds),
        "Val Recall": recall_score(y_val, val_preds),
        "Val F1-Score": f1_score(y_val, val_preds)
    })

results_df = pd.DataFrame(results).sort_values(by="Val F1-Score", ascending=False)
print("=== مقارنة النماذج على بيانات التحقق (Validation Set) ===")
print(results_df.to_string(index=False))

=== مقارنة النماذج على بيانات التحقق (Validation Set) ===
              Model  Val Accuracy  Val Precision  Val Recall  Val F1-Score
           LightGBM      0.848248       0.751334    0.672569      0.709773
Tuned Decision Tree      0.830257       0.723620    0.622541      0.669285
Logistic Regression      0.814206       0.702863    0.565767      0.626908
      Random Forest      0.805599       0.811869    0.384486      0.521839


In [14]:
best_model_name = results_df.iloc[0]["Model"]
best_pipeline = trained_pipelines[best_model_name]

print(f"\nأفضل نموذج تم اختياره: {best_model_name}")
test_preds = best_pipeline.predict(X_test)

print(classification_report(y_test, test_preds))


أفضل نموذج تم اختياره: LightGBM

=== التقييم النهائي على بيانات الاختبار (Test Set) ===
              precision    recall  f1-score   support

           0       0.88      0.91      0.90      9339
           1       0.75      0.67      0.71      3557

    accuracy                           0.85     12896
   macro avg       0.81      0.79      0.80     12896
weighted avg       0.84      0.85      0.85     12896



In [15]:
os.makedirs('models', exist_ok=True)
joblib.dump(best_pipeline, 'models/hotel_cancellation_model.pkl')

['models/hotel_cancellation_model.pkl']